# Drawing Modality — Spiral Dataset (Otsu + HOG + LBP + **XGBoost**)




## 0. Dépendances

In [8]:
import subprocess, sys

packages = [
    "numpy", "pandas", "scikit-learn", "scikit-image",
    "Pillow", "matplotlib", "scipy", "joblib",
    "xgboost", "shap",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("Tous les packages sont prêts.")


Tous les packages sont prêts.


## 1. Chemins et hyper-paramètres

In [9]:
from pathlib import Path
import numpy as np

REPO_ROOT   = Path('.').resolve().parent.parent
DATA_ROOT   = REPO_ROOT / 'data' / 'spiral'
MODEL_DIR   = REPO_ROOT / 'models'
MODEL_PATH  = MODEL_DIR / 'drawing_spiral_v3_hog_lbp_xgb.joblib'

IMG_SIZE         = (128, 128)
HOG_ORIENTATIONS = 9
HOG_PIXELS_CELL  = (8, 8)
HOG_CELLS_BLOCK  = (2, 2)
LBP_RADIUS       = 3
LBP_N_POINTS     = 8 * LBP_RADIUS
LBP_N_BINS       = 26

RANDOM_STATE  = 42
N_ITER_SEARCH = 20   # 20 itérations suffisent sur un petit dataset (300 fits → 100 fits)

for split in ('training', 'testing'):
    for cls in ('healthy', 'parkinson'):
        folder = DATA_ROOT / split / cls
        n = len(list(folder.glob('*.png')))
        print(f"{split}/{cls:>10} : {n} images")
print(f"\nREPO_ROOT : {REPO_ROOT}")
print(f"MODEL_PATH : {MODEL_PATH}")


training/   healthy : 36 images
training/ parkinson : 36 images
testing/   healthy : 15 images
testing/ parkinson : 15 images

REPO_ROOT : C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection
MODEL_PATH : C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\models\drawing_spiral_v3_hog_lbp_xgb.joblib


## 2. Chargement du dataset

Le split train/test est **pré-défini** par les auteurs du dataset.
Nous respectons cette séparation et n'appliquons l'augmentation **qu'au train**.


In [10]:
from PIL import Image


def load_split(split: str):
    """Charge toutes les images d'un split ('training' ou 'testing').

    Retourne (images_PIL, labels, subject_ids).
    Nom de fichier ex : V01HE02.png  →  sujet='V01', classe='HC' (H=healthy)
    """
    imgs, labs, sids = [], [], []
    for label, cls in enumerate(('healthy', 'parkinson')):
        folder = DATA_ROOT / split / cls
        for path in sorted(folder.glob('*.png')):
            sid = path.stem[:3]
            imgs.append(Image.open(path).convert('L'))
            labs.append(label)   # 0=HC, 1=PD
            sids.append(sid)
    return imgs, labs, sids


train_imgs, train_labs, train_sids = load_split('training')
test_imgs,  test_labs,  test_sids  = load_split('testing')

print(f"Train : {len(train_imgs)} images  ({sum(l==0 for l in train_labs)} HC / {sum(l==1 for l in train_labs)} PD)")
print(f"Test  : {len(test_imgs)} images  ({sum(l==0 for l in test_labs)} HC / {sum(l==1 for l in test_labs)} PD)")
print(f"Sujets train uniques : {sorted(set(train_sids))}")


Train : 72 images  (36 HC / 36 PD)
Test  : 30 images  (15 HC / 15 PD)
Sujets train uniques : ['V01', 'V02', 'V03', 'V04', 'V05', 'V06', 'V07', 'V08', 'V09', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V55']


## 3. Visualisation du dataset

In [11]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

hc_idx = [i for i, l in enumerate(train_labs) if l == 0]
pd_idx = [i for i, l in enumerate(train_labs) if l == 1]

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for col, idx in enumerate(hc_idx[:5]):
    axes[0, col].imshow(train_imgs[idx], cmap='gray')
    axes[0, col].set_title(f'HC — {train_sids[idx]}', fontsize=9)
    axes[0, col].axis('off')
for col, idx in enumerate(pd_idx[:5]):
    axes[1, col].imshow(train_imgs[idx], cmap='gray')
    axes[1, col].set_title(f'PD — {train_sids[idx]}', fontsize=9)
    axes[1, col].axis('off')

fig.suptitle('Exemples du train (haut : HC, bas : PD)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_ROOT / 'samples_train.png', dpi=80)
plt.show()
print('Figure sauvegardée.')


Figure sauvegardée.


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\3378507690.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Augmentation ×8 — géométrique + texture synthétique (train uniquement)

Le dataset train est très petit (72 images). On applique **8 transformations** par image :

- 4 rotations (0°, 90°, 180°, 270°)
- flip horizontal + flip vertical
- **flou gaussien léger** (radius=0.8)
- **filtre de netteté**


In [12]:
from PIL import ImageOps, ImageFilter

AUGMENTATIONS = [
    ('rot0',    lambda img: img),
    ('rot90',   lambda img: img.rotate(90, expand=False)),
    ('rot180',  lambda img: img.rotate(180, expand=False)),
    ('rot270',  lambda img: img.rotate(270, expand=False)),
    ('flipH',   lambda img: ImageOps.mirror(img)),
    ('flipV',   lambda img: ImageOps.flip(img)),
    ('blur',    lambda img: img.filter(ImageFilter.GaussianBlur(radius=0.8))),
    ('sharpen', lambda img: img.filter(ImageFilter.SHARPEN)),
]

aug_imgs, aug_labs, aug_sids = [], [], []
for img, lab, sid in zip(train_imgs, train_labs, train_sids):
    for aug_name, aug_fn in AUGMENTATIONS:
        aug_imgs.append(aug_fn(img))
        aug_labs.append(lab)
        aug_sids.append(sid)

print(f"Après augmentation : {len(aug_imgs)} images train  (×{len(AUGMENTATIONS)} par image)")
print(f"  HC : {sum(l==0 for l in aug_labs)}  |  PD : {sum(l==1 for l in aug_labs)}")


Après augmentation : 576 images train  (×8 par image)
  HC : 288  |  PD : 288


## 5. Prétraitement et extraction de features

### Binarisation d'Otsu (correction domain shift) + HOG + LBP


In [13]:
from skimage.feature import hog, local_binary_pattern
from skimage.filters import threshold_otsu


def preprocess_image(pil_img: Image.Image, target_size=IMG_SIZE) -> np.ndarray:
    img = pil_img.convert('L').resize(target_size, Image.LANCZOS)
    arr = np.array(img, dtype=np.float32) / 255.0
    thresh = threshold_otsu(arr)
    return (arr < thresh).astype(np.float32)


def extract_hog_features(arr: np.ndarray) -> np.ndarray:
    return hog(
        arr,
        orientations=HOG_ORIENTATIONS,
        pixels_per_cell=HOG_PIXELS_CELL,
        cells_per_block=HOG_CELLS_BLOCK,
        block_norm='L2-Hys',
        feature_vector=True,
    ).astype(np.float32)


def extract_lbp_features(arr: np.ndarray) -> np.ndarray:
    lbp = local_binary_pattern(arr, P=LBP_N_POINTS, R=LBP_RADIUS, method='uniform')
    hist, _ = np.histogram(lbp.ravel(), bins=LBP_N_BINS, range=(0, LBP_N_BINS))
    hist = hist.astype(np.float32)
    hist /= (hist.sum() + 1e-8)
    return hist


def extract_features(pil_img: Image.Image) -> np.ndarray:
    arr = preprocess_image(pil_img)
    return np.concatenate([extract_hog_features(arr), extract_lbp_features(arr)])


X_train = np.array([extract_features(img) for img in aug_imgs])
y_train = np.array(aug_labs)
g_train = np.array(aug_sids)

X_test = np.array([extract_features(img) for img in test_imgs])
y_test = np.array(test_labs)

hog_dim = extract_hog_features(preprocess_image(train_imgs[0])).shape[0]
print(f"X_train : {X_train.shape}  |  X_test : {X_test.shape}")
print(f"HOG : {hog_dim} features  |  LBP : {LBP_N_BINS} features  |  Total : {X_train.shape[1]}")


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(


X_train : (576, 8126)  |  X_test : (30, 8126)
HOG : 8100 features  |  LBP : 26 features  |  Total : 8126


## 6. Recherche d'hyperparamètres XGBoost

`RandomizedSearchCV` avec `StratifiedGroupKFold` pour que les augmentations d'un même
sujet restent **toujours dans le même pli** (pas de fuite de données).



In [14]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedGroupKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform
import pandas as pd

unique_subjects = np.unique(g_train)
n_splits = min(5, len(unique_subjects) // 2)
cv = StratifiedGroupKFold(n_splits=n_splits)
print(f"Validation croisée : {n_splits} plis, {len(unique_subjects)} sujets uniques")

# Ratio classes pour scale_pos_weight
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
scale_pos = n_neg / n_pos
print(f"scale_pos_weight de base : {scale_pos:.2f}  (neg={n_neg}, pos={n_pos})")

xgb_base = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', XGBClassifier(
        objective='binary:logistic',
        eval_metric='auc',
        tree_method='hist',   # algorithme histogramme : ~10x plus rapide que 'exact'
        device='cpu',
        random_state=RANDOM_STATE,
        n_jobs=1,             # 1 thread ici, parallélisme géré par RandomizedSearchCV
    )),
])

param_dist = {
    'clf__n_estimators':      randint(100, 400),
    'clf__max_depth':         randint(2, 7),
    'clf__learning_rate':     uniform(0.01, 0.29),
    'clf__subsample':         uniform(0.6, 0.4),
    'clf__colsample_bytree':  uniform(0.5, 0.5),
    'clf__min_child_weight':  randint(1, 10),
    'clf__gamma':             uniform(0.0, 0.5),
    'clf__reg_alpha':         uniform(0.0, 1.0),
    'clf__reg_lambda':        uniform(0.5, 2.0),
    'clf__scale_pos_weight':  [1.0, scale_pos, scale_pos * 0.5],
}

search = RandomizedSearchCV(
    xgb_base,
    param_distributions=param_dist,
    n_iter=N_ITER_SEARCH,
    scoring='roc_auc',
    cv=cv,
    refit=True,
    n_jobs=-1,           # parallélise les folds sur tous les cœurs disponibles
    random_state=RANDOM_STATE,
    verbose=2,
)

search.fit(X_train, y_train, groups=g_train)

print(f"\nMeilleur AUC CV : {search.best_score_:.4f}")
print("Meilleurs hyperparamètres :")
for k, v in search.best_params_.items():
    print(f"  {k:35s} = {v}")


Validation croisée : 5 plis, 16 sujets uniques
scale_pos_weight de base : 1.00  (neg=288, pos=288)
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Meilleur AUC CV : 0.8639
Meilleurs hyperparamètres :
  clf__colsample_bytree               = 0.7055185066591156
  clf__gamma                          = 0.016525366450274193
  clf__learning_rate                  = 0.11007066192773805
  clf__max_depth                      = 2
  clf__min_child_weight               = 3
  clf__n_estimators                   = 225
  clf__reg_alpha                      = 0.1448948720912231
  clf__reg_lambda                     = 1.478905520555126
  clf__scale_pos_weight               = 1.0
  clf__subsample                      = 0.6968221086046001


## 7. Visualisation des résultats de la recherche

In [15]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

cv_results = pd.DataFrame(search.cv_results_).sort_values('mean_test_score', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution des AUC CV
axes[0].hist(cv_results['mean_test_score'], bins=20, color='steelblue', edgecolor='black', linewidth=0.6)
axes[0].axvline(search.best_score_, color='darkorange', linestyle='--', linewidth=1.5,
                label=f'Meilleur = {search.best_score_:.3f}')
axes[0].set_xlabel('AUC CV (mean)', fontsize=11)
axes[0].set_ylabel('Fréquence', fontsize=11)
axes[0].set_title('Distribution des AUC — RandomizedSearch', fontsize=12, fontweight='bold')
axes[0].legend()

# Top 20 configurations
top20 = cv_results.head(20)
axes[1].barh(range(len(top20)), top20['mean_test_score'].values[::-1],
             xerr=top20['std_test_score'].values[::-1],
             color='steelblue', edgecolor='black', linewidth=0.6, capsize=3)
axes[1].set_xlim(0.5, 1.0)
axes[1].set_xlabel('AUC CV (mean ± std)', fontsize=11)
axes[1].set_title('Top 20 configurations', fontsize=12, fontweight='bold')
axes[1].set_yticks([])

plt.tight_layout()
plt.savefig(DATA_ROOT / 'xgb_search_results.png', dpi=80)
plt.show()
print('Figure sauvegardée.')


Figure sauvegardée.


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\927073966.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Recherche du seuil de décision optimal (OOF)

Prédictions **out-of-fold** avec le meilleur pipeline pour trouver le seuil
qui maximise le F1-score, sans regarder le test set.


In [16]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, f1_score, roc_auc_score
from sklearn.base import clone

best_pipe = search.best_estimator_

y_oof = np.zeros(len(y_train))
for tr_idx, val_idx in cv.split(X_train, y_train, g_train):
    fold_pipe = clone(best_pipe)
    fold_pipe.fit(X_train[tr_idx], y_train[tr_idx])
    y_oof[val_idx] = fold_pipe.predict_proba(X_train[val_idx])[:, 1]

fpr, tpr, thresholds = roc_curve(y_train, y_oof)
f1_scores_thresh = [f1_score(y_train, y_oof >= t, zero_division=0) for t in thresholds]
best_idx = int(np.argmax(f1_scores_thresh))
DECISION_THRESHOLD = float(thresholds[best_idx])

oof_auc = roc_auc_score(y_train, y_oof)
print(f"AUC OOF global   : {oof_auc:.4f}")
print(f"Seuil F1-optimal : {DECISION_THRESHOLD:.4f}")
print(f"F1 OOF @seuil    : {f1_scores_thresh[best_idx]:.4f}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(fpr, tpr, color='steelblue', linewidth=2, label=f'AUC = {oof_auc:.3f}')
ax1.plot([0, 1], [0, 1], '--', color='gray')
ax1.fill_between(fpr, tpr, alpha=0.1, color='steelblue')
ax1.set_xlabel('FPR'); ax1.set_ylabel('TPR')
ax1.set_title('Courbe ROC — OOF (XGBoost)', fontsize=12, fontweight='bold'); ax1.legend()

ax2.plot(thresholds, f1_scores_thresh, color='darkorange', linewidth=2)
ax2.axvline(DECISION_THRESHOLD, color='red', linestyle='--', linewidth=1.5,
            label=f'seuil optimal = {DECISION_THRESHOLD:.3f}')
ax2.set_xlabel('Seuil de décision'); ax2.set_ylabel('F1-score')
ax2.set_title('F1 vs seuil (OOF)', fontsize=12, fontweight='bold'); ax2.legend()
plt.tight_layout()
plt.savefig(DATA_ROOT / 'xgb_roc_threshold.png', dpi=80)
plt.show()


AUC OOF global   : 0.8522
Seuil F1-optimal : 0.3437
F1 OOF @seuil    : 0.7902


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\370960696.py:39: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Entraînement final sur tout le train augmenté

In [17]:
final_pipeline = clone(best_pipe)
final_pipeline.fit(X_train, y_train)

train_proba = final_pipeline.predict_proba(X_train)[:, 1]
train_auc   = roc_auc_score(y_train, train_proba)

print(f"Pipeline retenu : XGBoost (meilleurs hyperparamètres RandomizedSearch)")
print(f"AUC train (full) : {train_auc:.4f}")
print(f"AUC CV (mean)    : {search.best_score_:.4f}")
print(f"AUC OOF          : {oof_auc:.4f}")
print(f"Seuil retenu     : {DECISION_THRESHOLD:.4f}")
print()
print("Hyperparamètres XGBoost :")
for k, v in search.best_params_.items():
    print(f"  {k.replace('clf__', ''):30s} = {v}")


Pipeline retenu : XGBoost (meilleurs hyperparamètres RandomizedSearch)
AUC train (full) : 1.0000
AUC CV (mean)    : 0.8639
AUC OOF          : 0.8522
Seuil retenu     : 0.3437

Hyperparamètres XGBoost :
  colsample_bytree               = 0.7055185066591156
  gamma                          = 0.016525366450274193
  learning_rate                  = 0.11007066192773805
  max_depth                      = 2
  min_child_weight               = 3
  n_estimators                   = 225
  reg_alpha                      = 0.1448948720912231
  reg_lambda                     = 1.478905520555126
  scale_pos_weight               = 1.0
  subsample                      = 0.6968221086046001


## 10. Importance des features — gain XGBoost natif

XGBoost expose nativement trois métriques d'importance :
- `weight` : fréquence d'utilisation dans les arbres
- `gain` : amélioration moyenne du score à chaque split
- `cover` : nombre moyen d'échantillons couverts


In [18]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

xgb_clf = final_pipeline.named_steps['clf']
n_features = X_train.shape[1]

feature_names = (
    [f'HOG_{i}' for i in range(hog_dim)] +
    [f'LBP_{i}' for i in range(LBP_N_BINS)]
)

importance_types = ['weight', 'gain', 'cover']
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, imp_type in zip(axes, importance_types):
    scores = xgb_clf.get_booster().get_score(importance_type=imp_type)
    named_scores = {feature_names[int(k[1:])]: v for k, v in scores.items()}
    top_k = sorted(named_scores.items(), key=lambda x: x[1], reverse=True)[:20]
    names_top, vals_top = zip(*top_k)
    ax.barh(range(len(names_top)), list(reversed(vals_top)), color='steelblue', edgecolor='black', linewidth=0.5)
    ax.set_yticks(range(len(names_top)))
    ax.set_yticklabels(list(reversed(names_top)), fontsize=8)
    ax.set_title(f'Top 20 — {imp_type}', fontsize=11, fontweight='bold')
    ax.set_xlabel(imp_type.capitalize())

plt.suptitle('Importance des features XGBoost (HOG + LBP)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_ROOT / 'xgb_feature_importance.png', dpi=80)
plt.show()
print('Figure sauvegardée.')


Figure sauvegardée.


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\2774536121.py:30: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 11. Interprétabilité — SHAP values

SHAP (SHapley Additive exPlanations) fournit une explication **locale et globale**
des prédictions, indépendante du modèle.

- **Beeswarm plot** : chaque point est une observation ; couleur = valeur de la feature ; position x = impact sur le score.
- **Bar plot** : importance globale (|SHAP| moyen).


In [19]:
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

scaler = final_pipeline.named_steps['scaler']
X_train_scaled = scaler.transform(X_train)

explainer = shap.TreeExplainer(xgb_clf)
shap_values = explainer.shap_values(X_train_scaled)

print(f"SHAP values shape : {shap_values.shape}")

fig_bee, ax_bee = plt.subplots(figsize=(10, 7))
shap.summary_plot(
    shap_values,
    X_train_scaled,
    feature_names=feature_names,
    max_display=20,
    show=False,
    plot_type='dot',
)
plt.title('SHAP Beeswarm — Top 20 features (XGBoost)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_ROOT / 'xgb_shap_beeswarm.png', dpi=80, bbox_inches='tight')
plt.show()

fig_bar, ax_bar = plt.subplots(figsize=(10, 6))
shap.summary_plot(
    shap_values,
    X_train_scaled,
    feature_names=feature_names,
    max_display=20,
    show=False,
    plot_type='bar',
)
plt.title('SHAP Feature Importance (|SHAP| moyen)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig(DATA_ROOT / 'xgb_shap_bar.png', dpi=80, bbox_inches='tight')
plt.show()
print('Figures SHAP sauvegardées.')


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


SHAP values shape : (576, 8126)


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\2889791484.py:26: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


Figures SHAP sauvegardées.


C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\2889791484.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 12. Évaluation sur le jeu de test

Le test set contient des images **non augmentées, non vues à l'entraînement**.


In [20]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import (
    classification_report, roc_auc_score,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_curve,
)

y_test_proba = final_pipeline.predict_proba(X_test)[:, 1]
y_test_pred  = (y_test_proba >= DECISION_THRESHOLD).astype(int)

test_auc = roc_auc_score(y_test, y_test_proba)
print(f"AUC test : {test_auc:.4f}")
print(f"Seuil    : {DECISION_THRESHOLD:.4f}")
print()
print(classification_report(y_test, y_test_pred, target_names=['HC', 'PD']))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Matrice de confusion
cm = confusion_matrix(y_test, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['HC', 'PD'])
disp.plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title(f'Matrice de confusion — test\nAUC = {test_auc:.3f}', fontsize=11, fontweight='bold')

# Courbe ROC test
fpr_t, tpr_t, _ = roc_curve(y_test, y_test_proba)
axes[1].plot(fpr_t, tpr_t, color='darkorange', linewidth=2, label=f'AUC = {test_auc:.3f}')
axes[1].plot([0, 1], [0, 1], '--', color='gray')
axes[1].fill_between(fpr_t, tpr_t, alpha=0.1, color='darkorange')
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
axes[1].set_title('Courbe ROC — jeu de test', fontsize=11, fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.savefig(DATA_ROOT / 'xgb_confusion_roc_test.png', dpi=80)
plt.show()


AUC test : 0.7956
Seuil    : 0.3437

              precision    recall  f1-score   support

          HC       0.64      0.60      0.62        15
          PD       0.62      0.67      0.65        15

    accuracy                           0.63        30
   macro avg       0.63      0.63      0.63        30
weighted avg       0.63      0.63      0.63        30



C:\Users\adm.deskit\AppData\Local\Temp\ipykernel_36040\1488899482.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 13. Comparaison v2 vs v3



In [21]:
import joblib
import pandas as pd

v2_path = MODEL_DIR / 'drawing_spiral_v2_hog_lbp_pipeline.joblib'
comparison_rows = []

if v2_path.exists():
    v2 = joblib.load(v2_path)
    comparison_rows.append({
        'Version':    'v2 (SVM/RF/GBM best)',
        'Classifieur': v2.get('classifier', '—'),
        'AUC CV':     f"{v2.get('cv_auc_mean', float('nan')):.3f} ± {v2.get('cv_auc_std', float('nan')):.3f}",
        'AUC test':   f"{v2.get('test_auc', float('nan')):.3f}",
        'Seuil':      f"{v2.get('threshold', float('nan')):.3f}",
    })
else:
    print("Artefact v2 non trouvé, comparaison partielle.")

comparison_rows.append({
    'Version':    'v3 (XGBoost — ce notebook)',
    'Classifieur': f"XGBoost ({search.best_params_['clf__n_estimators']} trees, "
                   f"depth={search.best_params_['clf__max_depth']}, "
                   f"lr={search.best_params_['clf__learning_rate']:.3f})",
    'AUC CV':     f"{search.best_score_:.3f}",
    'AUC test':   f"{test_auc:.3f}",
    'Seuil':      f"{DECISION_THRESHOLD:.3f}",
})

df_cmp = pd.DataFrame(comparison_rows).set_index('Version')
print(df_cmp.to_string())


                                                       Classifieur         AUC CV AUC test  Seuil
Version                                                                                          
v2 (SVM/RF/GBM best)                                 Random Forest  0.871 ± 0.030    0.880  0.499
v3 (XGBoost — ce notebook)  XGBoost (225 trees, depth=2, lr=0.110)          0.864    0.796  0.344


## 14. Export de l'artefact



In [22]:
import joblib

MODEL_DIR.mkdir(parents=True, exist_ok=True)

xgb_params = {k.replace('clf__', ''): v for k, v in search.best_params_.items()}

artifact = {
    'pipeline':       final_pipeline,
    'hog_params': {
        'img_size':        IMG_SIZE,
        'orientations':    HOG_ORIENTATIONS,
        'pixels_per_cell': HOG_PIXELS_CELL,
        'cells_per_block': HOG_CELLS_BLOCK,
    },
    'lbp_params': {
        'radius':   LBP_RADIUS,
        'n_points': LBP_N_POINTS,
        'n_bins':   LBP_N_BINS,
    },
    'preprocessing':   'otsu_binarization',
    'feature_type':    'HOG+LBP_on_binary',
    'model':           'drawing_spiral_hog_lbp_xgboost_v3',
    'classifier':      'XGBoost',
    'xgb_params':      xgb_params,
    'threshold':       DECISION_THRESHOLD,
    'cv_auc_mean':     float(search.best_score_),
    'cv_auc_std':      float(cv_results.loc[cv_results['mean_test_score'] == search.best_score_, 'std_test_score'].values[0]),
    'oof_auc':         float(oof_auc),
    'test_auc':        float(test_auc),
    'n_iter_search':   N_ITER_SEARCH,
    'trained_on':      'kaggle_spiral_dataset_x8aug',
    'aug_factor':      8,
    'domain_shift':    'Otsu binarization + blur/sharpen aug to reduce paper-vs-canvas gap.',
    'note':            'Dataset Kaggle Spiral. Otsu + HOG 8x8 + LBP. XGBoost tuned via RandomizedSearchCV. '
                       'Prototype exploratoire — pas un dispositif médical.',
}

joblib.dump(artifact, MODEL_PATH)
print(f'Modèle sauvegardé : {MODEL_PATH}')

loaded = joblib.load(MODEL_PATH)
assert loaded['hog_params'] == artifact['hog_params']
assert loaded['preprocessing'] == 'otsu_binarization'
assert loaded['classifier'] == 'XGBoost'
print('Vérification OK.')
print(f"Classifieur  : {loaded['classifier']}")
print(f"Prétraitement: {loaded['preprocessing']}")
print(f"AUC CV       : {loaded['cv_auc_mean']:.4f}")
print(f"AUC OOF      : {loaded['oof_auc']:.4f}")
print(f"AUC test     : {loaded['test_auc']:.4f}")
print(f"Seuil        : {loaded['threshold']:.4f}")


Modèle sauvegardé : C:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\models\drawing_spiral_v3_hog_lbp_xgb.joblib
Vérification OK.
Classifieur  : XGBoost
Prétraitement: otsu_binarization
AUC CV       : 0.8639
AUC OOF      : 0.8522
AUC test     : 0.7956
Seuil        : 0.3437


## 15. Test d'inférence bout en bout

Simulation du flux réel : image PIL → buffer PNG → features → score XGBoost.


In [23]:
import io as _io

errors = 0
total  = 0

for label_name, src_imgs, src_labs in [
    ('HC', [img for img, l in zip(test_imgs, test_labs) if l == 0][:3], [0]*3),
    ('PD', [img for img, l in zip(test_imgs, test_labs) if l == 1][:3], [1]*3),
]:
    for img, true_lab in zip(src_imgs, src_labs):
        buf = _io.BytesIO()
        img.save(buf, format='PNG')
        received = Image.open(_io.BytesIO(buf.getvalue()))

        feats = extract_features(received).reshape(1, -1)
        score = loaded['pipeline'].predict_proba(feats)[0, 1]
        pred  = 'PD' if score >= loaded['threshold'] else 'HC'
        status = 'OK' if pred == label_name else 'ERREUR'
        if status == 'ERREUR':
            errors += 1
        total += 1
        print(f'[{status}] Vraie={label_name}  Score PD={score:.4f}  Prédit={pred}')

print(f"\nRésultat : {total - errors}/{total} corrects sur ce sous-ensemble de test")


[ERREUR] Vraie=HC  Score PD=0.7537  Prédit=PD
[ERREUR] Vraie=HC  Score PD=0.4106  Prédit=PD


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differe

[OK] Vraie=HC  Score PD=0.1365  Prédit=HC
[ERREUR] Vraie=PD  Score PD=0.1224  Prédit=HC
[OK] Vraie=PD  Score PD=0.9807  Prédit=PD
[OK] Vraie=PD  Score PD=0.9954  Prédit=PD

Résultat : 3/6 corrects sur ce sous-ensemble de test


c:\Users\adm.deskit\Desktop\github\multimodal-parkinson-detection\.venv\Lib\site-packages\skimage\feature\texture.py:385: UserWarning: Applying `local_binary_pattern` to floating-point images may give unexpected results when small numerical differences between adjacent pixels are present. It is recommended to use this function with images of integer dtype.
  warnings.warn(
